# Speech Extraction Pipeline — PhoWhisper-large (Kaggle T4×2)

Nhánh **speech** cho video retrieval tiếng Việt. Self-contained: chạy lần lượt từ trên xuống.

**Luồng:** video → ffmpeg(16kHz mono) → PhoWhisper-large (CTranslate2) → WhisperX forced-alignment (word timestamp) → lọc hallucination → JSON `{video_id, start, end, text, words[]}`.

**Trước khi chạy:**
1. Bật **Internet** (Settings → Internet on) để tải lib + model.
2. Chọn accelerator **GPU T4 × 2**.
3. Sửa `DATASET_GLOB` ở cell Config trỏ đúng dataset video của bạn.

**Checkpoint:** video đã có `.speech.json` sẽ được bỏ qua → hết 12h cứ chạy lại cell Run.


## 1. Config

In [ ]:
# >>> SỬA CHỖ NÀY cho đúng dataset của bạn <<<
DATASET_GLOB  = "/kaggle/input/**/*.mp4"   # ĐỆ QUY mọi dataset/thư mục con (sửa nếu cần)
WORK_DIR      = "/kaggle/working/speech_out"

NUM_GPUS      = 2          # T4×2
BEAM_SIZE     = 5          # cân bằng cho T4 (chất lượng ~ beam 10, nhanh hơn nhiều)
BATCH_SIZE    = 8          # batch WhisperX (an toàn VRAM 16GB)
COMPUTE_TYPE  = "float16"  # T4: float16 (KHÔNG bf16). int8_float16 nếu muốn nhanh hơn.
MIN_WORD_SCORE = 0.0       # 0.0 = TẮT lọc theo điểm align (an toàn). Nâng lên 0.30 SAU khi xem phân bố.

import os
os.makedirs(WORK_DIR, exist_ok=True)
print("Config OK ->", WORK_DIR)


## 2. Ghi file pipeline
Ghi `speech_pipeline.py` và `kaggle_run.py` (wrapper vá cuDNN cho CTranslate2).

In [ ]:
%%writefile speech_pipeline.py
"""
SOTA Speech Extraction Pipeline cho Video Retrieval (tiếng Việt)
Backbone ASR: PhoWhisper-large (vinai/PhoWhisper-large)

Backend (cùng model ASR, khác nguồn timestamp):
  - whisperx       : PhoWhisper-large (qua faster-whisper) + forced alignment wav2vec2-vi
                     -> word-timestamp CHÍNH XÁC (mặc định)
  - faster-whisper : PhoWhisper-large + timestamp cross-attention (nhẹ hơn, 1 model)

Chạy ĐA GPU: --num_gpus 2 -> chia video round-robin cho 2 worker, mỗi worker ghim 1 GPU.
Checkpoint  : tự bỏ qua video đã có file .speech.json (an toàn khi session Kaggle hết giờ).

Luồng mỗi video:
  video -> ffmpeg(16kHz mono) -> ASR(PhoWhisper-large) -> word-timestamp
        -> lọc hallucination -> chuẩn hóa -> JSON kèm timestamp (+ tùy chọn index Elasticsearch)

Lưu ý filter chất lượng (--min_word_score):
  Backend whisperx KHÔNG có avg_logprob/no_speech_prob; tín hiệu chất lượng duy nhất là
  điểm forced-alignment (avg_word_score). Nếu aligner lệch vocab -> mọi điểm thấp -> filter
  có thể xóa sạch output. Vì vậy MẶC ĐỊNH --min_word_score 0.0 (TẮT lọc theo điểm).
  Chạy thử 1 lần, xem phân bố avg_word_score trong JSON, rồi mới nâng threshold (vd 0.30).
"""

import os
import re
import json
import subprocess
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import List, Optional


# ============== SHIM torchaudio.AudioMetaData (whisperx -> pyannote.audio) ==============
# torchaudio mới đã gỡ thuộc tính top-level AudioMetaData; pyannote.audio (được import qua
# whisperx ở mức module, KỂ CẢ khi ta dùng silero VAD) vẫn tham chiếu nó -> AttributeError
# lúc import. Vá lại trước khi import whisperx cho an toàn (không ảnh hưởng faster-whisper).
def _patch_torchaudio_audiometadata():
    try:
        import torchaudio
    except Exception:
        return
    if hasattr(torchaudio, "AudioMetaData"):
        return
    import importlib
    amd = None
    for mod in ("torchaudio.backend.common", "torchaudio._backend.common"):
        try:
            amd = importlib.import_module(mod).AudioMetaData
            break
        except Exception:
            pass
    if amd is None:
        class amd:  # fallback: chỉ cần là attribute hợp lệ cho type-hint khi import
            pass
    torchaudio.AudioMetaData = amd


_patch_torchaudio_audiometadata()


# ============================== CONFIG ==============================
MODEL_DIR = "phowhisper-large-ct2"
ALIGN_MODEL = "nguyenvulebinh/wav2vec2-base-vietnamese-250h"
DEVICE = "cuda"
COMPUTE_TYPE = "float16"      # T4: float16 (đừng bf16). int8_float16 nếu cần nhanh hơn.
LANGUAGE = "vi"

# Decode mặc định (ưu tiên chất lượng). Trên T4 có thể hạ beam_size còn 5 cho nhanh.
ASR_OPTIONS = {
    "beam_size": 10,
    "best_of": 10,
    "temperatures": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    "condition_on_previous_text": False,
    "compression_ratio_threshold": 2.4,
    "log_prob_threshold": -1.0,
    "no_speech_threshold": 0.6,
}


# ============================== DATA MODEL ==============================
@dataclass
class Word:
    start: Optional[float]
    end: Optional[float]
    word: str
    score: float


@dataclass
class Segment:
    video_id: str
    start: float
    end: float
    text: str
    words: List[Word] = field(default_factory=list)
    avg_logprob: Optional[float] = None       # faster-whisper
    no_speech_prob: Optional[float] = None     # faster-whisper
    avg_word_score: Optional[float] = None     # whisperx (sau alignment)


# ============================== TÁCH AUDIO ==============================
def extract_audio(video_path: str, out_wav: str) -> str:
    cmd = ["ffmpeg", "-y", "-i", video_path, "-vn", "-ac", "1", "-ar", "16000",
           "-c:a", "pcm_s16le", out_wav]
    proc = subprocess.run(cmd, capture_output=True)
    if proc.returncode != 0:
        # giữ lại stderr để biết file nào hỏng / codec lạ (thay vì nuốt lỗi)
        tail = proc.stderr.decode("utf-8", "ignore")[-800:]
        raise RuntimeError(f"ffmpeg lỗi khi tách audio '{video_path}':\n{tail}")
    return out_wav


# ============================== EXTRACTOR (load model 1 lần / process) ==============================
class SpeechExtractor:
    def __init__(self, backend: str = "whisperx", device: str = DEVICE,
                 compute_type: str = COMPUTE_TYPE, beam_size: Optional[int] = None,
                 batch_size: int = 16, vad_method: str = "silero"):
        self.backend = backend
        self.device = device
        self.batch_size = batch_size

        # asr_options cục bộ (cho phép override beam_size khi chạy Kaggle)
        self.asr_opts = dict(ASR_OPTIONS)
        if beam_size is not None:
            self.asr_opts["beam_size"] = beam_size
            self.asr_opts["best_of"] = beam_size

        if backend == "faster-whisper":
            from faster_whisper import WhisperModel
            self.model = WhisperModel(MODEL_DIR, device=device, compute_type=compute_type)

        elif backend == "whisperx":
            import whisperx
            self.wx = whisperx
            # vad_method="silero" tránh model VAD pyannote bị gated. NHƯNG vài bản whisperx không có
            # tham số này -> fallback bỏ qua: dùng VAD mặc định mà whisperx tự host (không cần token).
            common = dict(compute_type=compute_type, language=LANGUAGE, asr_options=self.asr_opts)
            try:
                self.model = whisperx.load_model(MODEL_DIR, device, vad_method=vad_method, **common)
            except TypeError:
                self.model = whisperx.load_model(MODEL_DIR, device, **common)
            self.align_model, self.align_meta = whisperx.load_align_model(
                language_code=LANGUAGE, device=device, model_name=ALIGN_MODEL,
            )
        else:
            raise ValueError(f"backend không hợp lệ: {backend}")

    def _transcribe_fw(self, wav_path: str, video_id: str) -> List[Segment]:
        segments_gen, _ = self.model.transcribe(
            wav_path, language=LANGUAGE,
            beam_size=self.asr_opts["beam_size"], best_of=self.asr_opts["best_of"],
            temperature=self.asr_opts["temperatures"],
            condition_on_previous_text=self.asr_opts["condition_on_previous_text"],
            word_timestamps=True, vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=500, speech_pad_ms=200),
            compression_ratio_threshold=self.asr_opts["compression_ratio_threshold"],
            log_prob_threshold=self.asr_opts["log_prob_threshold"],
            no_speech_threshold=self.asr_opts["no_speech_threshold"],
        )
        out: List[Segment] = []
        for s in segments_gen:
            words = [Word(round(w.start, 3), round(w.end, 3), w.word.strip(), round(w.probability, 4))
                     for w in (s.words or [])]
            out.append(Segment(video_id, round(s.start, 3), round(s.end, 3), s.text.strip(),
                               words, avg_logprob=round(s.avg_logprob, 4),
                               no_speech_prob=round(s.no_speech_prob, 4)))
        return out

    def _transcribe_wx(self, wav_path: str, video_id: str) -> List[Segment]:
        audio = self.wx.load_audio(wav_path)
        result = self.model.transcribe(audio, batch_size=self.batch_size, language=LANGUAGE)
        result = self.wx.align(result["segments"], self.align_model, self.align_meta,
                               audio, self.device, return_char_alignments=False)
        out: List[Segment] = []
        for s in result["segments"]:
            words, scores = [], []
            for w in s.get("words", []):
                sc = float(w.get("score", 0.0)); scores.append(sc)
                words.append(Word(w.get("start"), w.get("end"), w.get("word", "").strip(), round(sc, 4)))
            avg_ws = round(sum(scores) / len(scores), 4) if scores else None
            out.append(Segment(video_id, round(float(s["start"]), 3), round(float(s["end"]), 3),
                               s["text"].strip(), words, avg_word_score=avg_ws))
        return out

    def transcribe(self, wav_path: str, video_id: str) -> List[Segment]:
        return self._transcribe_fw(wav_path, video_id) if self.backend == "faster-whisper" \
            else self._transcribe_wx(wav_path, video_id)


# ============================== LỌC HALLUCINATION ==============================
HALLUCINATION_PATTERNS = [
    r"cảm ơn các bạn đã (theo dõi|xem|lắng nghe)",
    r"(hãy|nhớ) (đăng ký|like|subscribe|ấn chuông)",
    r"ghiền mì gõ",
    r"^[\.\-–—\s]+$",
]


def is_hallucination(seg: Segment, min_word_score: float = 0.0) -> bool:
    t = seg.text.lower().strip()
    if not t:
        return True
    if seg.no_speech_prob is not None and seg.avg_logprob is not None:
        if seg.no_speech_prob > 0.7 and seg.avg_logprob < -1.0:
            return True
    # Lọc theo điểm align CHỈ khi min_word_score > 0 (tránh xóa sạch nếu aligner lệch vocab).
    if min_word_score > 0.0 and seg.avg_word_score is not None and seg.avg_word_score < min_word_score:
        return True
    words = t.split()
    if len(words) >= 4 and len(set(words)) <= 2:
        return True
    return any(re.search(p, t) for p in HALLUCINATION_PATTERNS)


def clean(segments: List[Segment], min_word_score: float = 0.0) -> List[Segment]:
    return [s for s in segments if not is_hallucination(s, min_word_score)]


# ============================== CHUẨN HÓA ==============================
def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


# ============================== INDEX ELASTICSEARCH (tùy chọn) ==============================
ES_INDEX = "speech_segments"
ES_MAPPING = {
    "settings": {"analysis": {"analyzer": {
        "vi_analyzer": {"type": "custom", "tokenizer": "vi_tokenizer", "filter": ["lowercase"]}}}},
    "mappings": {"properties": {
        "video_id": {"type": "keyword"}, "start": {"type": "float"}, "end": {"type": "float"},
        "text": {"type": "text", "analyzer": "vi_analyzer"}, "text_raw": {"type": "text"}}},
}


def build_index(es):
    if not es.indices.exists(index=ES_INDEX):
        es.indices.create(index=ES_INDEX, body=ES_MAPPING)


def index_segments(es, segments: List[Segment]):
    from elasticsearch.helpers import bulk
    actions = [{"_index": ES_INDEX, "_id": f"{s.video_id}_{i}",
                "_source": {"video_id": s.video_id, "start": s.start, "end": s.end,
                            "text": s.text, "text_raw": s.text}}
               for i, s in enumerate(segments)]
    if actions:
        bulk(es, actions)


# ============================== XỬ LÝ 1 VIDEO ==============================
def process_video(extractor: SpeechExtractor, video_path: str, work_dir: str,
                  es=None, min_word_score: float = 0.0) -> List[Segment]:
    video_id = Path(video_path).stem
    wav = os.path.join(work_dir, f"{video_id}.wav")
    extract_audio(video_path, wav)

    segs = clean(extractor.transcribe(wav, video_id), min_word_score)
    for s in segs:
        s.text = normalize_text(s.text)

    out_json = os.path.join(work_dir, f"{video_id}.speech.json")
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump([asdict(s) for s in segs], f, ensure_ascii=False, indent=2)

    if es is not None:
        index_segments(es, segs)
    try:
        os.remove(wav)
    except OSError:
        pass
    return segs


# ============================== WORKER ĐA GPU ==============================
def _worker(gpu_id: int, videos: List[str], backend: str, work_dir: str,
            es_host: Optional[str], compute_type: str,
            beam_size: Optional[int], batch_size: int, min_word_score: float):
    # PHẢI set trước khi torch/cuda khởi tạo trong process con -> "cuda" == GPU vật lý gpu_id
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

    extractor = SpeechExtractor(backend=backend, device="cuda", compute_type=compute_type,
                                beam_size=beam_size, batch_size=batch_size)
    es = None
    if es_host:
        from elasticsearch import Elasticsearch
        es = Elasticsearch(es_host)

    for v in videos:
        vid = Path(v).stem
        if os.path.exists(os.path.join(work_dir, f"{vid}.speech.json")):
            print(f"[gpu{gpu_id}] skip {vid} (đã có)", flush=True)
            continue
        try:
            n = len(process_video(extractor, v, work_dir, es, min_word_score))
            print(f"[gpu{gpu_id}] {vid} -> {n} segs", flush=True)
        except Exception as e:
            print(f"[gpu{gpu_id}] LỖI {vid}: {e}", flush=True)


def run_multi_gpu(videos, num_gpus, backend, work_dir, es_host,
                  compute_type, beam_size, batch_size, min_word_score):
    import multiprocessing as mp
    if es_host:  # tạo index 1 lần ở tiến trình cha, tránh race
        from elasticsearch import Elasticsearch
        build_index(Elasticsearch(es_host))

    shards = [videos[i::num_gpus] for i in range(num_gpus)]
    ctx = mp.get_context("spawn")  # bắt buộc cho CUDA
    procs = []
    for gid in range(num_gpus):
        if not shards[gid]:
            continue
        p = ctx.Process(target=_worker, args=(gid, shards[gid], backend, work_dir,
                                              es_host, compute_type, beam_size, batch_size,
                                              min_word_score))
        p.start(); procs.append(p)
    for p in procs:
        p.join()


# ============================== MAIN ==============================
if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser(description="SOTA Vietnamese speech extraction (PhoWhisper-large)")
    ap.add_argument("--videos", nargs="+", required=True)
    ap.add_argument("--backend", choices=["whisperx", "faster-whisper"], default="whisperx")
    ap.add_argument("--num_gpus", type=int, default=1)
    ap.add_argument("--work_dir", default="./speech_out")
    ap.add_argument("--es_host", default=None)
    ap.add_argument("--compute_type", default=COMPUTE_TYPE)
    ap.add_argument("--beam_size", type=int, default=None, help="override (T4 nên dùng 5)")
    ap.add_argument("--batch_size", type=int, default=16, help="batch WhisperX")
    ap.add_argument("--min_word_score", type=float, default=0.0,
                    help="lọc segment có avg_word_score < ngưỡng (whisperx). 0.0 = TẮT (an toàn). "
                         "Xem phân bố score rồi mới nâng (vd 0.30).")
    args = ap.parse_args()

    os.makedirs(args.work_dir, exist_ok=True)

    if args.num_gpus > 1:
        run_multi_gpu(args.videos, args.num_gpus, args.backend, args.work_dir, args.es_host,
                      args.compute_type, args.beam_size, args.batch_size, args.min_word_score)
    else:
        extractor = SpeechExtractor(backend=args.backend, compute_type=args.compute_type,
                                    beam_size=args.beam_size, batch_size=args.batch_size)
        es = None
        if args.es_host:
            from elasticsearch import Elasticsearch
            es = Elasticsearch(args.es_host); build_index(es)
        for v in args.videos:
            vid = Path(v).stem
            if os.path.exists(os.path.join(args.work_dir, f"{vid}.speech.json")):
                print(f"skip {vid} (đã có)"); continue
            n = len(process_video(extractor, v, args.work_dir, es, args.min_word_score))
            print(f"[{vid}] backend={args.backend} -> {n} segs")


In [ ]:
%%writefile kaggle_run.py
"""
Wrapper chạy speech_pipeline.py trên Kaggle.
Vá lỗi kinh điển 'libcudnn_*.so not found' của CTranslate2: nạp LD_LIBRARY_PATH trỏ tới
các thư viện CUDA của pip (nvidia-*-cu12) NGAY KHI khởi chạy tiến trình python
— phải set lúc launch, không set được sau khi process đã start.

Lưu ý: KHÔNG dùng nvidia.cudnn.__file__ — trên Kaggle 'nvidia.cudnn' có thể là namespace
package -> __file__ == None -> os.path.dirname(None) ném TypeError. Dùng __path__ / glob.

Dùng:
    !python kaggle_run.py --videos /kaggle/input/<ds>/*.mp4 \
        --num_gpus 2 --work_dir /kaggle/working/speech_out \
        --beam_size 5 --batch_size 8
Mọi tham số được forward thẳng sang speech_pipeline.py.
"""
import os
import sys
import glob
import subprocess

# Gom mọi thư mục nvidia/*/lib (cudnn, cublas, ...) mà CTranslate2 cần.
lib_dirs = []
try:
    import nvidia  # namespace package gốc của các wheel nvidia-*-cu12
    for base in list(getattr(nvidia, "__path__", []) or []):
        lib_dirs += glob.glob(os.path.join(base, "*", "lib"))
except Exception:
    pass
for base in sys.path:                       # fallback: quét site-packages trên sys.path
    if base and os.path.isdir(base):
        lib_dirs += glob.glob(os.path.join(base, "nvidia", "*", "lib"))

# lọc thư mục tồn tại + dedupe giữ thứ tự
seen = set()
lib_dirs = [d for d in lib_dirs if os.path.isdir(d) and not (d in seen or seen.add(d))]

env = dict(os.environ)
if lib_dirs:
    env["LD_LIBRARY_PATH"] = ":".join(lib_dirs) + ":" + env.get("LD_LIBRARY_PATH", "")
    print(f"[kaggle_run] +{len(lib_dirs)} nvidia lib dir(s) vào LD_LIBRARY_PATH")
    for d in lib_dirs:
        print("   ", d)
else:
    print("[kaggle_run] CẢNH BÁO: không thấy nvidia/*/lib — kiểm tra đã pip install nvidia-cudnn-cu12 chưa.")

cmd = [sys.executable, "speech_pipeline.py"] + sys.argv[1:]
print("[kaggle_run]", " ".join(cmd[:2]), "...")
sys.exit(subprocess.run(cmd, env=env).returncode)


## 3. Setup
Cài lib (pin `whisperx` để tránh vỡ API), rồi convert PhoWhisper-large → CTranslate2 (một lần).

In [ ]:
# === Pin stack tương thích cho whisperx (Kaggle torchaudio quá mới làm vỡ pyannote) ===
# torch/torchaudio/torchvision = 2.2.2/2.2.2/0.17.2 (triple khớp), pyannote 3.3.2,
# ctranslate2 4.x (cuDNN 9, khác soname cuDNN 8 của torch). numpy<2 vì torch 2.2 cần numpy 1.
# transformers==4.48.3: bản mới chặn torch.load (CVE-2025-32434) nếu torch<2.6 -> aligner .bin
# không load được; 4.48.3 chưa có cái chặn đó.
!pip -q install "numpy<2" torch==2.2.2 torchaudio==2.2.2 torchvision==0.17.2
!pip -q install "whisperx==3.3.1" "pyannote.audio==3.3.2" "ctranslate2<4.5" \
    transformers==4.48.3 nvidia-cudnn-cu12 elasticsearch
!pip -q install "numpy<2"   # ép lại phòng khi dep kéo numpy lên 2 (gây lỗi ABI _ARRAY_API)
!python -c "import torch,torchaudio,torchvision,numpy as np; print('torch',torch.__version__,'| ta',torchaudio.__version__,'| tv',torchvision.__version__,'| numpy',np.__version__)"
!pip show whisperx 2>/dev/null | grep -i '^Version:' | sed 's/^/whisperx /'
print(">>> Cài lib xong. Restart kernel rồi chạy lại từ cell này.")


In [ ]:
import os, subprocess
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
# (tùy chọn) đặt token để hết cảnh báo rate-limit / tải nhanh hơn:
# os.environ["HF_TOKEN"] = "hf_xxx"

# KHUYẾN NGHỊ cho job nhiều session: pre-stage model đã convert thành Kaggle Dataset rồi copy về,
# vừa nhanh vừa khỏi đụng HF mỗi session:
#   !cp -r /kaggle/input/phowhisper-ct2/phowhisper-large-ct2 .
if not os.path.isdir("phowhisper-large-ct2"):
    print(">>> Convert PhoWhisper-large -> CT2 (lần đầu, ~vài phút). Chờ tới khi in 'Convert xong'.")
    r = subprocess.run([
        "ct2-transformers-converter",
        "--model", "vinai/PhoWhisper-large",
        "--output_dir", "phowhisper-large-ct2",
        "--copy_files", "tokenizer.json", "preprocessor_config.json",
        "--quantization", "float16",
    ], capture_output=True, text=True)
    # Traceback 'Thread-auto_conversion ... 403 ... Discussions are disabled' là của background
    # thread transformers (thử tạo bản safetensors trên Hub) -> VÔ HẠI, không chặn convert.
    # Chỉ coi là lỗi khi thư mục output không được tạo:
    if not os.path.isdir("phowhisper-large-ct2"):
        print(r.stdout[-2000:]); print(r.stderr[-3000:])
        raise RuntimeError("Convert CT2 thất bại — xem log phía trên.")
    print(">>> Convert xong:", sorted(os.listdir("phowhisper-large-ct2")))
else:
    print(">>> phowhisper-large-ct2 đã có, bỏ qua convert")


## (tùy chọn) Ước lượng tổng thời gian chạy
Benchmark pipeline trên 2 video mẫu để tách **thời gian load model** khỏi **throughput thật** (realtime-factor), rồi suy ra tổng giờ + số session 12h cần cho cả corpus. Tốn ~vài phút.

Chạy **sau Setup** (đã có lib + model + file `.py`). Bỏ qua nếu không cần.

In [ ]:
# ===== ƯỚC LƯỢNG TỔNG THỜI GIAN (phương pháp 2-lần-chạy) =====
#   runA = [A ngắn]        -> tA = load + xử_lý(A)
#   runB = [A ngắn, B vừa] -> tB = load + xử_lý(A) + xử_lý(B)
#   => xử_lý(B) = tB - tA  => realtime-factor (xRT) trên 1 GPU, không lẫn thời gian load.
import os, glob, time, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor

videos = sorted(glob.glob(DATASET_GLOB, recursive=True))
assert videos, "Không thấy video — kiểm tra DATASET_GLOB."

def ffdur(p):
    out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", p],
        capture_output=True, text=True).stdout.strip()
    return float(out) if out else 0.0

print(f"Quét độ dài {len(videos)} video...")
with ThreadPoolExecutor(max_workers=16) as ex:
    durs = list(ex.map(ffdur, videos))
total_audio_s = sum(durs)

# A = video ngắn (P10), B = video vừa (P50) -> delta đủ ổn định mà không tốn quá nhiều
order = sorted(range(len(videos)), key=lambda i: durs[i])
A = videos[order[int(0.10 * (len(order) - 1))]]
B = videos[order[int(0.50 * (len(order) - 1))]]
durA, durB = ffdur(A), ffdur(B)
print(f"Mẫu A (ngắn): {os.path.basename(A)}  {durA/60:.1f} phút")
print(f"Mẫu B (vừa) : {os.path.basename(B)}  {durB/60:.1f} phút")

def bench(vids, d):
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
    t = time.time()
    r = subprocess.run(["python", "kaggle_run.py", "--videos", *vids, "--backend", "whisperx",
        "--num_gpus", "1", "--work_dir", d, "--beam_size", str(BEAM_SIZE),
        "--batch_size", str(BATCH_SIZE), "--compute_type", COMPUTE_TYPE],
        capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:]); print(r.stderr[-1500:]); raise RuntimeError("benchmark lỗi")
    return time.time() - t

print("\n>>> runA (load + A)...")
tA = bench([A], "/kaggle/working/_benchA")
print(f"    tA = {tA:.1f}s")
print(">>> runB (load + A + B)...")
tB = bench([A, B], "/kaggle/working/_benchB")
print(f"    tB = {tB:.1f}s")

delta = tB - tA                       # ~ thời gian xử lý riêng video B
assert delta > 0, "delta <= 0 (nhiễu đo) — chạy lại hoặc chọn B dài hơn."
xrt = durB / delta                    # audio-giây / wall-giây, trên 1 GPU
load_s = max(0.0, tA - durA / xrt)    # thời gian load model + tải aligner (1 lần)

per_gpu_audio = total_audio_s / NUM_GPUS
est_wall_s = load_s + per_gpu_audio / xrt
SESS_H = 11.0                         # giờ hữu dụng / session (chừa lề cho 12h)
n_sessions = est_wall_s / 3600 / SESS_H

def hm(s):
    m = int(s // 60); h, m = divmod(m, 60); return f"{h}h{m:02d}m"

print("\n===== ƯỚC LƯỢNG =====")
print(f"Tổng audio          : {total_audio_s/3600:.1f} giờ ({len(videos)} video)")
print(f"Tốc độ (1 GPU)      : {xrt:.1f}x realtime  (beam={BEAM_SIZE}, batch={BATCH_SIZE}, {COMPUTE_TYPE})")
print(f"Load model (1 lần)  : {load_s:.0f}s")
print(f"Ước lượng wall ({NUM_GPUS} GPU): {hm(est_wall_s)}")
print(f"Số session 12h cần  : ~{n_sessions:.1f}  (tính {SESS_H:.0f}h hữu dụng/session)")
if xrt < 8:
    print("→ Chậm: cân nhắc COMPUTE_TYPE='int8_float16' (T4 chạy int8 tốt) để tăng tốc.")

shutil.rmtree("/kaggle/working/_benchA", ignore_errors=True)
shutil.rmtree("/kaggle/working/_benchB", ignore_errors=True)


## 4. Run
Chạy đa GPU qua `kaggle_run.py` (set `LD_LIBRARY_PATH` cuDNN trước khi launch). Glob mở rộng trong Python để chắc ăn.

In [ ]:
import glob, subprocess
videos = sorted(glob.glob(DATASET_GLOB, recursive=True))
print(f"Tìm thấy {len(videos)} video từ {DATASET_GLOB}")
assert videos, "Không thấy video nào — kiểm tra lại DATASET_GLOB."

cmd = ["python", "kaggle_run.py",
       "--videos", *videos,
       "--backend", "whisperx",
       "--num_gpus", str(NUM_GPUS),
       "--work_dir", WORK_DIR,
       "--beam_size", str(BEAM_SIZE),
       "--batch_size", str(BATCH_SIZE),
       "--compute_type", COMPUTE_TYPE,
       "--min_word_score", str(MIN_WORD_SCORE)]
print(" ".join(cmd[:2]), "...", len(videos), "videos")
subprocess.run(cmd, check=True)


## 5. Kiểm tra output (QUAN TRỌNG)
Xác nhận pipeline không xóa nhầm segment. Xem **phân bố `avg_word_score`**: speech sạch thường rơi vào ~0.4–0.9.
- Nếu phần lớn điểm < 0.2 → aligner wav2vec2-vi đang **lệch vocab/dấu** → ĐỪNG bật `MIN_WORD_SCORE`.
- Nếu điểm hợp lý → có thể đặt `MIN_WORD_SCORE = 0.30` ở cell Config rồi chạy lại để lọc hallucination chặt hơn.


In [ ]:
import json, glob, os, statistics
files = sorted(glob.glob(os.path.join(WORK_DIR, "*.speech.json")))
print(f"{len(files)} file .speech.json")

scores, total = [], 0
for f in files:
    segs = json.load(open(f, encoding="utf-8"))
    total += len(segs)
    scores += [s["avg_word_score"] for s in segs if s.get("avg_word_score") is not None]

print(f"Tổng segment: {total}")
if scores:
    qs = statistics.quantiles(scores, n=10)
    print("avg_word_score deciles:", [round(x, 3) for x in qs])
    print("min / median / max   :", round(min(scores), 3),
          round(statistics.median(scores), 3), round(max(scores), 3))
else:
    print("Không có avg_word_score (backend faster-whisper hoặc 0 segment).")

# Xem thử vài segment đầu của file đầu tiên
if files:
    segs = json.load(open(files[0], encoding="utf-8"))
    print(f"\n--- Mẫu: {os.path.basename(files[0])} ---")
    for s in segs[:8]:
        print(f"[{s['start']:6.1f}-{s['end']:6.1f}] score={s.get('avg_word_score')}  {s['text']}")
